# Linea de Muerte Z

##  1. Big Picture

* Preprocesamiento
   * Se quitan del datataset los *envenenados* atributos  mprestamos_personales y cprestamos_personales
   * Se agregan lags y delta_lags de orden 1 y 2
   * pongo en NULL las var que en un mes tienen todo 0
   * invierto el valor de internet a partir de 202010
   * IPC
   * SAC
   
* Modelado no se optimizan hiperarámetros
* Produccion
   * Entrenamieento final
      * Se entrena en {202101, 202102, 202103, 202104}
      * Se hace un conservador undersampling de **0.50** de los "CONTINUA"
      * POS = {"BAJA+1", "BAJA+2"}
      * librería  *zLightGBM*
         * **100** canaritos se agregan al comienzo del dataset, porque ahora hay mas datos
         * gradient_bound se deja en su default de  **0.1**
   * Clasificacion
      * Se corta en 11000  envios

---
Resultados :
* ganancia de 391.085 M en el Private Leaderboard ( 11.000 en el Public )
* utiliza 17 GB de memoria RAM
* corre en 30 minutos

## Inicializacion

In [1]:
# limpio la memoria
Sys.time()
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

[1] "2025-11-11 04:26:25 UTC"

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,657976,35.2,1454708,77.7,1276591,68.2
Vcells,1221411,9.4,8388608,64.0,1975153,15.1


In [2]:
PARAM <- list()
PARAM$experimento <- "z50_valid06_FE_nan_02"
PARAM$semilla_primigenia <- 974411

In [3]:
setwd("/content/buckets/b1/exp")
experimento_folder <- PARAM$experimento
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

## Preprocesamiento

### Generacion de la clase_ternaria

In [4]:
Sys.time()
require( "data.table" )

# leo el dataset
dataset <- fread("~/buckets/b1/datasets/competencia_02_crudo.csv.gz" )

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
  "pos" = .I,
  numero_de_cliente,
  periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 )
]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
  shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente
]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
  ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
  clase_ternaria := "BAJA+1"
]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
  & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
  clase_ternaria := "BAJA+2"
]

# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

rm(dsimple)
gc()
Sys.time()

[1] "2025-11-11 04:26:26 UTC"

Loading required package: data.table



,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,766818,41.0,1454708,77.7,1454708,77.7
Vcells,722144220,5509.6,1017394361,7762.2,845998590,6454.5


[1] "2025-11-11 04:27:02 UTC"

### Eliminacion de Features

La auténtica  **salsa mágica**  de este script es la eliminación de estos dos atributos del dataset ya que tran problemas con el Data Drifting
* mprestamos_personales
* cprestamos_personales

el problema con estos campos se detectó manualmente mediante un analisis exploratorio de datos y análisis de corridas de LightGBM en meses previos.

In [5]:
dataset[, mprestamos_personales := NULL ]
dataset[, cprestamos_personales := NULL ]

ncol(dataset)
Sys.time()

[1] 153

[1] "2025-11-11 04:27:02 UTC"

### Data Quality

Se deben reparar los atributos del dataset que para un cierto mes TODOS sus valores son cero.
Relevar en forma muy minuciosa en el dataset cuales son los <atributo,mes> que estan dañados.
Algunas alternativas de solución son:

No hacer absolutamente nada, dejar el valor 0 tal cual está, a sabiendas que es incorrecto
Reemplazar esos valores dañados por NA
Interpolar cada valor dañado por el valor del mes previo y el posterior
Calcularlo a partir de un modelo, libreria MICE
a este codigo de Data Quality lo debera escribir usted

In [6]:
library(data.table)
stopifnot(is.data.table(dataset))

id_cols  <- c("numero_de_cliente","foto_mes","clase_ternaria")
num_cols <- setdiff(names(dataset), id_cols)
num_cols <- num_cols[vapply(dataset[, ..num_cols], is.numeric, logical(1))]

# (mes, columna) con TODO 0 (estricto)
zero_map <- dataset[
  , lapply(.SD, function(v) isTRUE(all(v == 0))), 
  by = .(foto_mes), .SDcols = num_cols
]

zero_long <- melt(
  zero_map, id.vars = "foto_mes",
  variable.name = "columna", value.name = "all_zero"
)[all_zero == TRUE, .(foto_mes, columna)]

#  convertir la columna de nombres a character (evita el factor-gate)
zero_long[, columna := as.character(columna)]

if (nrow(zero_long) == 0L) {
  message("No hay (mes, columna) con TODO 0 estricto. Nada para NA-izar.")
} else {
  setorder(zero_long, foto_mes, columna)
  for (i in seq_len(nrow(zero_long))) {
    m  <- zero_long$foto_mes[i]
    cn <- zero_long$columna[i]  # <- ahora es character

    idx <- which(dataset$foto_mes == m)

    # preservar tipo
    if (is.integer(dataset[[cn]])) {
      set(dataset, i = idx, j = cn, value = NA_integer_)
    } else {
      set(dataset, i = idx, j = cn, value = NA_real_)
    }
  }

  # log amigable
  cat("Reemplacé por NA las columnas con TODO 0 (estricto):\n")
  print(zero_long[, .(columnas = toString(columna)), by = foto_mes][order(foto_mes)])
}


ncol(dataset)
Sys.time()

Reemplacé por NA las columnas con TODO 0 (estricto):
   foto_mes
      <int>
1:   201904
2:   201905
3:   201910
4:   202002
5:   202006
6:   202009
7:   202010
8:   202102
9:   202105
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

[1] 153

[1] "2025-11-11 04:27:25 UTC"

In [7]:
# Corregir inversión de valores en campo 'internet' a partir de 202010

# 1) Limpieza: cualquier cosa que no sea 0/1 -> NA (en todos los meses)
dataset[, internet := fifelse(internet %in% 0:1, as.integer(internet), NA_integer_)]

# 2) Flip: solo para meses anteriores a 202010 y con valor válido
dataset[ foto_mes < 202010 & !is.na(internet), internet := 1L - internet ]

ncol(dataset)
Sys.time()

[1] 153

[1] "2025-11-11 04:27:25 UTC"

### Data Drifting

Se debe corregir el drifting natural que ocurre en loa datos, en particular los datos monetarios que se vieron fuertemente afectados por una alta inflación
Posibles métodos son:

No hacer absolutamente nada
Ajuste de valores monetarios por indices del tipo :
IPC Indice de Precios al Consumidor
Dolar Oficial
Dolar Blue
UVA Unidad de Valor Adquisitivo
a este codigo de Data Drifting lo debera escribir usted

#### IPC

In [8]:
library(data.table)

# ================= IPC desde variaciones mensuales =================
# Variaciones mensuales (%) en orden 201901, 201902, ..., 202108  (32 valores)
ipc_var <- c(
  2.9, 3.8, 4.7, 3.4, 3.1, 2.7, 2.2, 4.0, 5.9, 3.3, 4.3, 3.7,
  2.3, 2.0, 3.3, 1.5, 1.5, 2.2, 1.9, 2.7, 2.8, 3.8, 3.2, 4.0,
  4.0, 3.6, 4.8, 4.1, 3.3, 3.2, 3.0, 2.5
)

# Genero la secuencia de foto_mes 201901..202108
seq_fm <- function(from_fm, to_fm){
  dseq <- seq(as.Date(paste0(from_fm,"01"), "%Y%m%d"),
              as.Date(paste0(to_fm,"01"), "%Y%m%d"), by="month")
  as.integer(format(dseq, "%Y%m"))
}
fm_seq <- seq_fm(201901L, 202108L)

stopifnot(length(ipc_var) == length(fm_seq))

ipc <- data.table(
  foto_mes = fm_seq,
  variacion_mensual = ipc_var
)

# Índice acumulado (base libre). Pongo 100 en el primer mes y acumulo.
ipc[, indice := 100 * cumprod(1 + variacion_mensual/100)]

# ================= Deflactar a una base elegida =================
base_mes <- 202108L            # sugerido: el último mes disponible
stopifnot(base_mes %in% ipc$foto_mes)

# Traigo el índice al dataset y calculo factor relativo a la base
setkey(ipc, foto_mes)
stopifnot("foto_mes" %in% names(dataset))
dataset <- ipc[dataset, on="foto_mes"]

indice_base <- ipc[foto_mes == base_mes, indice][1]
dataset[, defl_factor := indice / indice_base]  # >1 = precios más altos que la base

# Detecto columnas de montos (m* y Master_m*/Visa_m*)
montos_m_prefix <- grep("^m", names(dataset), value = TRUE)
montos_card     <- grep("^(Master|Visa)_m", names(dataset), value = TRUE)
montos_all <- unique(c(montos_m_prefix, montos_card))
montos_all <- montos_all[vapply(dataset[, ..montos_all], is.numeric, logical(1))]

# Deflactar IN-PLACE a pesos de 'base_mes'
for (cn in montos_all) {
  dataset[, (cn) := get(cn) / pmax(defl_factor, 1e-12)]
}

# (opcional) trazabilidad y limpieza
dataset[, ipc_base := base_mes]
dataset[, c("variacion_mensual","indice","defl_factor") := NULL]

cat(sprintf("IPC aplicado (base %d). Columnas deflactadas: %d\n", base_mes, length(montos_all)))

ncol(dataset)
Sys.time()

IPC aplicado (base 202108). Columnas deflactadas: 72


[1] 154

[1] "2025-11-11 04:27:48 UTC"

# SAC

In [ ]:
# ====== AJUSTE SAC CON REGLAS POR INCREMENTO DE TRX ======
library(data.table)
setDT(dataset)
setorder(dataset, numero_de_cliente, foto_mes)

# ---------- Parámetros ----------
SAC_MODE <- "ma3"      # "ma3" (media móvil 3m previa) o "factor"
SAC_FACTOR <- 1.5      # si usás "factor": sueldo observado / SAC_FACTOR  (=> SAC ~ sueldo/1.5 - sueldo)
EPS <- 1e-9

# ---------- Flags de calendario ----------
dataset[, kmes := foto_mes %% 100L]
dataset[, is_sac_month := as.integer(kmes %in% c(6L, 12L))]
dataset[, is_post_sac  := as.integer(kmes %in% c(7L, 1L))]

# ---------- Lags necesarios ----------
lag1 <- function(x) shift(x, 1L)

dataset[, `:=`(
  cpayroll_trx_lag1  = lag1(cpayroll_trx),
  cpayroll2_trx_lag1 = lag1(cpayroll2_trx),
  mpayroll_lag1      = lag1(mpayroll),
  mpayroll2_lag1     = lag1(mpayroll2)
), by = numero_de_cliente]

# ---------- Disparadores por stream ----------
dataset[, sac_trigger1 := as.integer(is_sac_month==1L &
                                     !is.na(cpayroll_trx) & !is.na(cpayroll_trx_lag1) &
                                     cpayroll_trx >= (cpayroll_trx_lag1 + 1L))]
dataset[, sac_trigger2 := as.integer(is_sac_month==1L &
                                     !is.na(cpayroll2_trx) & !is.na(cpayroll2_trx_lag1) &
                                     cpayroll2_trx >= (cpayroll2_trx_lag1 + 1L))]

# ---------- Base de sueldo y monto SAC por stream ----------
if (SAC_MODE == "ma3") {
  dataset[, base1 := frollmean(lag1(mpayroll),  3L, align="right", na.rm=TRUE), by=numero_de_cliente]
  dataset[, base2 := frollmean(lag1(mpayroll2), 3L, align="right", na.rm=TRUE), by=numero_de_cliente]
  dataset[, sac1  := 0.5 * pmax(0, fcoalesce(base1, 0))]
  dataset[, sac2  := 0.5 * pmax(0, fcoalesce(base2, 0))]
} else if (SAC_MODE == "factor") {
  # sueldo base ~ sueldo_observado / 1.5  → SAC ~ 0.5 * base
  dataset[, sac1 := 0.5 * pmax(0, fcoalesce(mpayroll, 0) / SAC_FACTOR)]
  dataset[, sac2 := 0.5 * pmax(0, fcoalesce(mpayroll2,0) / SAC_FACTOR)]
} else {
  stop("SAC_MODE debe ser 'ma3' o 'factor'")
}

# Solo aplicamos montos si hubo trigger ese mes (sino, 0)
dataset[, sac1_eff := fifelse(sac_trigger1==1L, sac1, 0)]
dataset[, sac2_eff := fifelse(sac_trigger2==1L, sac2, 0)]
dataset[, sac_total_eff := pmax(0, sac1_eff + sac2_eff)]

# ---------- 1) Deflactar mpayroll/mpayroll2 en junio/diciembre ----------
dataset[is_sac_month==1L & sac1_eff>0, mpayroll  := pmax(0, fcoalesce(mpayroll, 0)  - sac1_eff)]
dataset[is_sac_month==1L & sac2_eff>0, mpayroll2 := pmax(0, fcoalesce(mpayroll2, 0) - sac2_eff)]

# ---------- 2) Descontar a mcuentas_saldo en junio/diciembre ----------
dataset[is_sac_month==1L & sac_total_eff>0,
        mcuentas_saldo := fcoalesce(mcuentas_saldo, 0) - sac_total_eff]

# ---------- 3) Descontar arrastre en caja en julio y diciembre ----------
# Arrastre de junio: usar sac_total_eff del mes anterior (lag1)
dataset[, sac_total_eff_lag1 := lag1(sac_total_eff), by=numero_de_cliente]

# Julio (kmes==7): restar el SAC de junio (lag1); Diciembre (kmes==12): restar el actual
#dataset[kmes==7L & is.finite(sac_total_eff_lag1) & sac_total_eff_lag1>0, `:=`(
#  mcaja_ahorro           = fcoalesce(mcaja_ahorro, 0)           - sac_total_eff_lag1,
#  mcaja_ahorro_adicional = fcoalesce(mcaja_ahorro_adicional, 0) - sac_total_eff_lag1
#)]
#dataset[kmes==12L & is.finite(sac_total_eff) & sac_total_eff>0, `:=`(
#  mcaja_ahorro           = fcoalesce(mcaja_ahorro, 0)           - sac_total_eff,
#  mcaja_ahorro_adicional = fcoalesce(mcaja_ahorro_adicional, 0) - sac_total_eff
#)]

# ---------- 4) Restar 1 a conteos de payroll cuando hubo trigger ----------
dataset[is_sac_month==1L & sac_trigger1==1L & !is.na(cpayroll_trx),
        cpayroll_trx := pmax(0L, as.integer(round(cpayroll_trx)) - 1L)]
dataset[is_sac_month==1L & sac_trigger2==1L & !is.na(cpayroll2_trx),
        cpayroll2_trx := pmax(0L, as.integer(round(cpayroll2_trx)) - 1L)]

# ---------- 5) Ratios normalizados (para modelar, no pisamos las originales) ----------
#norm_ratio <- function(x) {
#  base <- frollmean(shift(x, 1L), 3L, align="right", na.rm=TRUE)
#  x / pmax(EPS, base)
#}
#if ("ctarjeta_debito_transacciones" %in% names(dataset)) {
#  dataset[, ctarjeta_debito_transacciones_ratio_ma3 :=
#            norm_ratio(ctarjeta_debito_transacciones), by=numero_de_cliente]
#}
#if ("mautoservicio" %in% names(dataset)) {
#  dataset[, mautoservicio_ratio_ma3 :=
#            norm_ratio(mautoservicio), by=numero_de_cliente]
#}

# ---------- Limpieza mínima ----------
aux <- c("base1","base2","sac1","sac2","sac1_eff","sac2_eff","sac_total_eff","sac_total_eff_lag1",
         "cpayroll_trx_lag1","cpayroll2_trx_lag1","mpayroll_lag1","mpayroll2_lag1")
aux <- intersect(aux, names(dataset))
if (length(aux)) dataset[, (aux) := NULL]

# ---------- Sanity log ----------
cat(
  "SAC aplicado.\n",
  "Triggers payroll1:", dataset[sac_trigger1==1L, .N], "| payroll2:", dataset[sac_trigger2==1L, .N], "\n",
  "Meses SAC afectados:", dataset[is_sac_month==1L & (sac_trigger1==1L | sac_trigger2==1L), uniqueN(foto_mes)], "\n"
)
# ====== /AJUSTE SAC ======


SAC aplicado.
 Triggers payroll1: 234086 | payroll2: 920 
 Meses SAC afectados: 5 


In [10]:
# BORRADO A MANO, UNA POR UNA las var creadas para el SAC
if ("kmes" %in% names(dataset))                         dataset[, kmes := NULL]
#if ("is_sac_month" %in% names(dataset))                 dataset[, is_sac_month := NULL]
#if ("is_post_sac" %in% names(dataset))                  dataset[, is_post_sac := NULL]

if ("cpayroll_trx_lag1" %in% names(dataset))            dataset[, cpayroll_trx_lag1 := NULL]
if ("cpayroll2_trx_lag1" %in% names(dataset))           dataset[, cpayroll2_trx_lag1 := NULL]
if ("mpayroll_lag1" %in% names(dataset))                dataset[, mpayroll_lag1 := NULL]
if ("mpayroll2_lag1" %in% names(dataset))               dataset[, mpayroll2_lag1 := NULL]

if ("sac_trigger1" %in% names(dataset))                 dataset[, sac_trigger1 := NULL]
if ("sac_trigger2" %in% names(dataset))                 dataset[, sac_trigger2 := NULL]

if ("base1" %in% names(dataset))                        dataset[, base1 := NULL]
if ("base2" %in% names(dataset))                        dataset[, base2 := NULL]
if ("sac1" %in% names(dataset))                         dataset[, sac1 := NULL]
if ("sac2" %in% names(dataset))                         dataset[, sac2 := NULL]
if ("sac1_eff" %in% names(dataset))                     dataset[, sac1_eff := NULL]
if ("sac2_eff" %in% names(dataset))                     dataset[, sac2_eff := NULL]
if ("sac_total_eff" %in% names(dataset))                dataset[, sac_total_eff := NULL]
if ("sac_total_eff_lag1" %in% names(dataset))           dataset[, sac_total_eff_lag1 := NULL]

# Por si agregaste ratios (comentá si no aplica)
if ("ctarjeta_debito_transacciones_ratio_ma3" %in% names(dataset))
  dataset[, ctarjeta_debito_transacciones_ratio_ma3 := NULL]
if ("mautoservicio_ratio_ma3" %in% names(dataset))
  dataset[, mautoservicio_ratio_ma3 := NULL]

cat("Listo. Columnas auxiliares ejecutadas al paredón.\n")
ncol(dataset)
Sys.time()

Listo. Columnas auxiliares ejecutadas al paredón.


[1] 156

[1] "2025-11-11 04:28:45 UTC"

# Grafico

In [11]:
library(data.table)
library(ggplot2)

stopifnot(exists("dataset"))
stopifnot("foto_mes" %in% names(dataset))

# -------- 1) Variables a graficar --------
vars_num <- names(dataset)[vapply(dataset, is.numeric, logical(1))]
vars_excluir <- c("foto_mes","numero_de_cliente","periodo0","periodo1","periodo2")
vars_plot <- setdiff(vars_num, vars_excluir)

# -------- 2) Promedios por foto_mes --------
promedios <- dataset[, lapply(.SD, function(x) mean(x, na.rm = TRUE)),
                     by = .(foto_mes), .SDcols = vars_plot]

# Largo + fecha + flag de aguinaldo (mes 06 o 12)
long <- melt(promedios, id.vars = "foto_mes",
             variable.name = "variable", value.name = "promedio")
long[, fecha := as.Date(paste0(foto_mes, "01"), "%Y%m%d")]
long[, mes   := as.integer(format(fecha, "%m"))]
long[, es_aguinaldo := mes %in% c(6L, 12L)]

# quitamos variables sin datos (todo NA)
val_count <- long[, .(all_na = all(is.na(promedio))), by = variable]
vars_plot <- val_count[all_na == FALSE, variable]
long <- long[variable %in% vars_plot]

cat(sprintf("Se graficarán %d variables en %d períodos.\n",
            length(vars_plot), long[, uniqueN(foto_mes)]))

# -------- 3) PDF: una variable por página --------
pdf("promedios_por_variable_por_mes_pretty.pdf", width = 14, height = 7)

for (v in vars_plot) {
  dtv <- long[variable == v][order(fecha)]

  p <- ggplot(dtv, aes(x = fecha, y = promedio)) +
    # línea base
    geom_line(na.rm = TRUE, linewidth = 0.5, alpha = 0.5) +
    # puntos normales
    geom_point(data = dtv[es_aguinaldo == FALSE],
               size = 2, na.rm = TRUE) +
    # puntos destacados (junio/diciembre)
    geom_point(data = dtv[es_aguinaldo == TRUE],
               aes(color = es_aguinaldo), size = 3, na.rm = TRUE) +
    scale_color_manual(values = c(`TRUE` = "#D81B60"),
                       labels = c(`TRUE` = "Junio/Diciembre"),
                       guide = guide_legend(override.aes = list(size = 4))) +
    scale_x_date(date_breaks = "1 month", date_labels = "%Y-%m",
                 expand = expansion(mult = c(0.01, 0.02))) +
    labs(
      title = paste0("Promedio mensual: ", v),
      subtitle = "Puntos en rojo = meses de aguinaldo (06 y 12)",
      x = "Período (YYYY-MM)",
      y = "Promedio",
      color = NULL
    ) +
    theme_minimal(base_size = 13) +
    theme(
      plot.title = element_text(face = "bold"),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 9),
      panel.grid.minor = element_blank(),
      panel.grid.major.x = element_line(linewidth = 0.2),
      plot.margin = margin(10, 20, 10, 20),
      legend.position = "top"
    )

  print(p)
}

dev.off()
cat("PDF generado: 'promedios_por_variable_por_mes_pretty.pdf'\n")


Se graficarán 153 variables en 32 períodos.


agg_record_1984768634 
                    2

PDF generado: 'promedios_por_variable_por_mes_pretty.pdf'


### Feature Engineering Intra-Mes

Crear variables nuevas a partir de las existentes dentro del mismo registro, **sin** ir a buscar información histórica.
<br> El siguiente código es un mínimo ejemplo, agregar nuevos features a gusto

In [12]:
# el mes 1,2, ..12 , podria servir para detectar estacionalidad
dataset[, kmes := foto_mes %% 100]

# creo un ctr_quarter que tenga en cuenta cuando
# los clientes hace 3 menos meses que estan
# ya que seria injusto considerar las transacciones medidas en menor tiempo
dataset[, ctrx_quarter_normalizado := as.numeric(ctrx_quarter) ]
dataset[cliente_antiguedad == 1, ctrx_quarter_normalizado := ctrx_quarter * 5.0]
dataset[cliente_antiguedad == 2, ctrx_quarter_normalizado := ctrx_quarter * 2.0]
dataset[cliente_antiguedad == 3, ctrx_quarter_normalizado := ctrx_quarter * 1.2]

# variable extraida de una tesis de maestria de Irlanda, se perdió el link
dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

ncol(dataset)
Sys.time()


[1] 159

[1] "2025-11-11 04:29:52 UTC"

### Feature Engineering Historico

In [13]:
if( !require("Rcpp")) install.packages("Rcpp", repos = "http://cran.us.r-project.org")
require("Rcpp")

Loading required package: Rcpp



In [14]:
# se calculan para los 6 meses previos el minimo, maximo y
#  tendencia calculada con cuadrados minimos
# la formula de calculo de la tendencia puede verse en
#  https://stats.libretexts.org/Bookshelves/Introductory_Statistics/Book%3A_Introductory_Statistics_(Shafer_and_Zhang)/10%3A_Correlation_and_Regression/10.04%3A_The_Least_Squares_Regression_Line
# para la maxíma velocidad esta funcion esta escrita en lenguaje C,
# y no en la porqueria de R o Python

cppFunction("NumericVector fhistC(NumericVector pcolumna, IntegerVector pdesde )
{
  /* Aqui se cargan los valores para la regresion */
  double  x[100] ;
  double  y[100] ;

  int n = pcolumna.size();
  NumericVector out( 5*n );

  for(int i = 0; i < n; i++)
  {
    //lag
    if( pdesde[i]-1 < i )  out[ i + 4*n ]  =  pcolumna[i-1] ;
    else                   out[ i + 4*n ]  =  NA_REAL ;


    int  libre    = 0 ;
    int  xvalor   = 1 ;

    for( int j= pdesde[i]-1;  j<=i; j++ )
    {
       double a = pcolumna[j] ;

       if( !R_IsNA( a ) )
       {
          y[ libre ]= a ;
          x[ libre ]= xvalor ;
          libre++ ;
       }

       xvalor++ ;
    }

    /* Si hay al menos dos valores */
    if( libre > 1 )
    {
      double  xsum  = x[0] ;
      double  ysum  = y[0] ;
      double  xysum = xsum * ysum ;
      double  xxsum = xsum * xsum ;
      double  vmin  = y[0] ;
      double  vmax  = y[0] ;

      for( int h=1; h<libre; h++)
      {
        xsum  += x[h] ;
        ysum  += y[h] ;
        xysum += x[h]*y[h] ;
        xxsum += x[h]*x[h] ;

        if( y[h] < vmin )  vmin = y[h] ;
        if( y[h] > vmax )  vmax = y[h] ;
      }

      out[ i ]  =  (libre*xysum - xsum*ysum)/(libre*xxsum -xsum*xsum) ;
      out[ i + n ]    =  vmin ;
      out[ i + 2*n ]  =  vmax ;
      out[ i + 3*n ]  =  ysum / libre ;
    }
    else
    {
      out[ i       ]  =  NA_REAL ;
      out[ i + n   ]  =  NA_REAL ;
      out[ i + 2*n ]  =  NA_REAL ;
      out[ i + 3*n ]  =  NA_REAL ;
    }
  }

  return  out;
}")

In [15]:
# calcula la tendencia de las variables cols de los ultimos 6 meses
# la tendencia es la pendiente de la recta que ajusta por cuadrados minimos
# La funcionalidad de ratioavg es autoria de  Daiana Sparta,  UAustral  2021

TendenciaYmuchomas <- function(
    dataset, cols, ventana = 6, tendencia = TRUE,
    minimo = TRUE, maximo = TRUE, promedio = TRUE,
    ratioavg = FALSE, ratiomax = FALSE) {
  gc(verbose= FALSE)
  # Esta es la cantidad de meses que utilizo para la historia
  ventana_regresion <- ventana

  last <- nrow(dataset)

  # creo el vector_desde que indica cada ventana
  # de esta forma se acelera el procesamiento ya que lo hago una sola vez
  vector_ids <- dataset[ , numero_de_cliente ]

  vector_desde <- seq(
    -ventana_regresion + 2,
    nrow(dataset) - ventana_regresion + 1
  )

  vector_desde[1:ventana_regresion] <- 1

  for (i in 2:last) {
    if (vector_ids[i - 1] != vector_ids[i]) {
      vector_desde[i] <- i
    }
  }
  for (i in 2:last) {
    if (vector_desde[i] < vector_desde[i - 1]) {
      vector_desde[i] <- vector_desde[i - 1]
    }
  }

  for (campo in cols) {
    nueva_col <- fhistC(dataset[, get(campo)], vector_desde)

    if (tendencia) {
      dataset[, paste0(campo, "_tend", ventana) :=
        nueva_col[(0 * last + 1):(1 * last)]]
    }

    if (minimo) {
      dataset[, paste0(campo, "_min", ventana) :=
        nueva_col[(1 * last + 1):(2 * last)]]
    }

    if (maximo) {
      dataset[, paste0(campo, "_max", ventana) :=
        nueva_col[(2 * last + 1):(3 * last)]]
    }

    if (promedio) {
      dataset[, paste0(campo, "_avg", ventana) :=
        nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratioavg) {
      dataset[, paste0(campo, "_ratioavg", ventana) :=
        get(campo) / nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratiomax) {
      dataset[, paste0(campo, "_ratiomax", ventana) :=
        get(campo) / nueva_col[(2 * last + 1):(3 * last)]]
    }
  }
}

In [16]:
# Feature Engineering Historico
# Creacion de LAGs
setorder(dataset, numero_de_cliente, foto_mes)

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
  colnames(dataset),
  c("numero_de_cliente", "foto_mes", "clase_ternaria")
))

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
  paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
  by= numero_de_cliente,
  .SDcols= cols_lagueables
]

# lags de orden 2
dataset[,
  paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
  by= numero_de_cliente,
  .SDcols= cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
  dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
  dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

Sys.time()

[1] "2025-11-11 04:31:55 UTC"

In [17]:
# parametros de Feature Engineering Historico de Tendencias
PARAM$FE_hist$Tendencias$run <- TRUE
PARAM$FE_hist$Tendencias$ventana <- 6
PARAM$FE_hist$Tendencias$tendencia <- TRUE
PARAM$FE_hist$Tendencias$minimo <- FALSE
PARAM$FE_hist$Tendencias$maximo <- FALSE
PARAM$FE_hist$Tendencias$promedio <- TRUE
PARAM$FE_hist$Tendencias$ratioavg <- TRUE
PARAM$FE_hist$Tendencias$ratiomax <- TRUE

In [18]:
# aqui se agregan las tendencias de los ultimos 6 meses

cols_lagueables <- intersect(cols_lagueables, colnames(dataset))
setorder(dataset, numero_de_cliente, foto_mes)

if( PARAM$FE_hist$Tendencias$run) {
    TendenciaYmuchomas(dataset,
    cols = cols_lagueables,
    ventana = PARAM$FE_hist$Tendencias$ventana, # 6 meses de historia
    tendencia = PARAM$FE_hist$Tendencias$tendencia,
    minimo = PARAM$FE_hist$Tendencias$minimo,
    maximo = PARAM$FE_hist$Tendencias$maximo,
    promedio = PARAM$FE_hist$Tendencias$promedio,
    ratioavg = PARAM$FE_hist$Tendencias$ratioavg,
    ratiomax = PARAM$FE_hist$Tendencias$ratiomax
  )
}

ncol(dataset)
Sys.time()

[1] 1179

[1] "2025-11-11 04:34:36 UTC"

### Feature Engineering a partir de hojas de Random Forest

In [19]:
if (!requireNamespace("lightgbm", quietly = TRUE)) {
  install.packages("lightgbm")  # si falla, usá la Opción B
}
library(lightgbm)

AgregaVarRandomForest <- function() {

  cat( "inicio AgregaVarRandomForest()\n")
  gc(verbose= FALSE)
  dataset[, clase01 := 0L ]
  dataset[ clase_ternaria %in% c( "BAJA+2", "BAJA+1"),
      clase01 := 1L ]

  campos_buenos <- setdiff(
    colnames(dataset),
    c( "clase_ternaria", "clase01")
  )

  dataset[, entrenamiento :=
    as.integer( foto_mes %in% PARAM$FE_rf$train$training )]

  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[entrenamiento == TRUE, campos_buenos, with = FALSE]),
    label = dataset[entrenamiento == TRUE, clase01],
    free_raw_data = FALSE
  )

  modelo <- lgb.train(
     data = dtrain,
     param = PARAM$FE_rf$lgb_param,
     verbose = -100
  )

  cat( "Fin construccion RandomForest\n" )
  # grabo el modelo, achivo .model
  lgb.save(modelo, file="modelo.model" )

  qarbolitos <- copy(PARAM$FE_rf$lgb_param$num_iterations)

  periodos <- dataset[ , unique( foto_mes ) ]

  for( periodo in  periodos )
  {
    cat( "periodo = ", periodo, "\n" )
    datamatrix <- data.matrix(dataset[ foto_mes== periodo, campos_buenos, with = FALSE])

    cat( "Inicio prediccion\n" )
    prediccion <- predict(
        modelo,
        datamatrix,
        type = "leaf"
    )
    cat( "Fin prediccion\n" )

    for( arbolito in 1:qarbolitos )
    {
       cat( arbolito, " " )
       hojas_arbol <- unique(prediccion[ , arbolito])

       for (pos in 1:length(hojas_arbol)) {
         # el numero de nodo de la hoja, estan salteados
         nodo_id <- hojas_arbol[pos]
         dataset[ foto_mes== periodo, paste0(
            "rf_", sprintf("%03d", arbolito),
             "_", sprintf("%03d", nodo_id)
          ) :=  as.integer( nodo_id == prediccion[ , arbolito]) ]

       }

       rm( hojas_arbol )
    }
    cat( "\n" )

    rm( prediccion )
    rm( datamatrix )
    gc(verbose= FALSE)
  }

  gc(verbose= FALSE)

  # borro clase01 , no debe ensuciar el dataset
  dataset[ , clase01 := NULL ]

}

In [20]:
# Parametros de Feature Engineering  a partir de hojas de Random Forest

# Estos CUATRO parametros son los que se deben modificar
PARAM$FE_rf$arbolitos= 20
PARAM$FE_rf$hojas_por_arbol= 16
PARAM$FE_rf$datos_por_hoja= 100
PARAM$FE_rf$mtry_ratio= 0.2

# Estos son quasi fijos
PARAM$FE_rf$train$training <- c( 202101, 202102, 202103)

# Estos TAMBIEN son quasi fijos
PARAM$FE_rf$lgb_param <-list(
    # parametros que se pueden cambiar
    num_iterations = PARAM$FE_rf$arbolitos,
    num_leaves  = PARAM$FE_rf$hojas_por_arbol,
    min_data_in_leaf = PARAM$FE_rf$datos_por_hoja,
    feature_fraction_bynode  = PARAM$FE_rf$mtry_ratio,

    # para que LightGBM emule Random Forest
    boosting = "rf",
    bagging_fraction = ( 1.0 - 1.0/exp(1.0) ),
    bagging_freq = 1.0,
    feature_fraction = 1.0,

    # genericos de LightGBM
    max_bin = 31L,
    objective = "binary",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    force_row_wise = TRUE,
    verbosity = -100,
    max_depth = -1L,
    min_gain_to_split = 0.0,
    min_sum_hessian_in_leaf = 0.001,
    lambda_l1 = 0.0,
    lambda_l2 = 0.0,

    pos_bagging_fraction = 1.0,
    neg_bagging_fraction = 1.0,
    is_unbalance = FALSE,
    scale_pos_weight = 1.0,

    drop_rate = 0.1,
    max_drop = 50,
    skip_drop = 0.5,

    extra_trees = FALSE
  )

In [ ]:
# Feature Engineering agregando variables de Random Forest
#  aqui es donde se hace el trabajo
AgregaVarRandomForest()

Sys.time()

### Reduccion dimensionalidad con canaritos asesinos

Intencionalmente este código no se brinda
<br> El objetivo de esto es reducir la dimensionalidad del dataset unicamente para que la Bayesian Optimization corra mas rápido
<br> Tambien se puede probar con Boruta

In [ ]:
# ====== REDUCCION DIMENSIONAL: BORUTA (después de AgregaVarRandomForest) ======
if (!requireNamespace("Boruta", quietly = TRUE)) install.packages("Boruta")
if (!requireNamespace("randomForest", quietly = TRUE)) install.packages("randomForest")
library(Boruta)
library(randomForest)
library(data.table)

set.seed(PARAM$semilla_primigenia)

# 1) Tomo filas de training para estimar importancia
boruta_rows <- dataset[ foto_mes %in% PARAM$train_final$meses ]

# 2) Target binario como factor para randomForest/Boruta
boruta_rows[, clase01 := 0L ]
boruta_rows[ clase_ternaria %in% c("BAJA+1","BAJA+2"), clase01 := 1L ]
boruta_rows[, clase01 := factor(ifelse(clase01 == 1L, "POS", "NEG"),
                                levels = c("NEG","POS"))]

# 3) Submuestreo: todas las POS (BAJA+1, BAJA+2) + 20% de CONTINUA
idx_pos <- which(boruta_rows$clase01 == "POS")
idx_neg <- which(boruta_rows$clase01 == "NEG")
n_take  <- max(1L, floor(0.20 * length(idx_neg)))
idx_neg_sample <- if (length(idx_neg) > 0L) sample(idx_neg, n_take) else integer(0)
idx_sel <- c(idx_pos, idx_neg_sample)

boruta_sub <- boruta_rows[idx_sel]

# 4) Candidatas: todas menos PK/fecha/target; sólo numéricas
id_cols  <- c("numero_de_cliente","foto_mes","clase_ternaria","clase01")
cand     <- setdiff(colnames(boruta_sub), id_cols)
num_cand <- cand[vapply(boruta_sub[, ..cand], is.numeric, logical(1))]

X_b <- copy(boruta_sub[, ..num_cand])
y_b <- boruta_sub$clase01

# 5) Imputación simple: reemplazo TODOS los NA por 0 (sin medianas)
for (j in seq_along(X_b)) {
  v <- X_b[[j]]
  if (anyNA(v)) set(X_b, which(is.na(v)), j, 0)
}

cat(sprintf("Boruta: %d filas (POS=%d, NEG=%d), %d columnas candidatas.\n",
            nrow(X_b), sum(y_b=="POS"), sum(y_b=="NEG"), ncol(X_b)))

# 6) Boruta (con randomForest)
bor <- Boruta(
  x = as.data.frame(X_b),
  y = y_b,
  maxRuns = 100,         # podés subirlo si querés más estabilidad
  pValue  = 0.01,
  mcAdj   = TRUE,
  doTrace = 1
)

# 7) Resolver tentativas
bor_fix <- TentativeRoughFix(bor)
sel_confirmed <- getSelectedAttributes(bor_fix, withTentative = FALSE)

cat(sprintf("Boruta confirmó %d variables (de %d candidatas).\n",
            length(sel_confirmed), length(num_cand)))

# 8) Persisto lista para auditoría
fwrite(data.table(feature = sel_confirmed),
       file = "boruta_features_confirmed.txt", sep = "\t")

# 9) Recorte del dataset completo ANTES de entrenar LGBM
keep_cols <- c("numero_de_cliente","foto_mes","clase_ternaria", sel_confirmed)
keep_cols <- intersect(keep_cols, colnames(dataset))

drop_cols <- setdiff(colnames(dataset), keep_cols)
if (length(drop_cols)) dataset[, (drop_cols) := NULL]

cat(sprintf("Dataset recortado: %d columnas retenidas.\n", ncol(dataset)))
# ====== /BORUTA ======


In [ ]:
ncol(dataset)
colnames(dataset)

## Modelado

### Optimizacion de Hipeparámetros

Este script no hace optimización de hiperparámetros
<br> **No** se llama a una Bayesian Optimization, ese paso se saltea.
<br> No hace falta hacer particiones <train, validate, test>,  tampoco se hace un k-fold cross validation

## Valid

In [ ]:
PARAM$qcanaritos <- 100

PARAM$lgbm <-  list(
  boosting= "gbdt",
  objective= "binary",
  metric= "custom",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE,
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_bin= 31L,
  min_data_in_leaf= 20L,  #este ya es el valor default de LightGBM

  num_iterations= 9999L, # dejo libre la cantidad de arboles, zLightGBM se detiene solo
  num_leaves= 999L, # dejo libre la cantidad de hojas, zLightGBM sabe cuando no hacer un split
  learning_rate= 1.0,  # se lo deja en 1.0 para que si el score esta por debajo de gradient_bound no se lo escale
    
  feature_fraction= 0.50, # un valor equilibrado, habra que probar alternativas ...
    
  canaritos= PARAM$qcanaritos, # fundamental en zLightGBM, aqui esta el control del overfitting
  gradient_bound= 0.1  # default de zLightGBM
)

In [ ]:
correr_zlgbm <- function(
    dataset_func, params, nombre) {

        # preparo el dataset de entrenamiento con undersampling
        set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
        dataset_func[, azar := runif(nrow(dataset_func))]
        dataset_func[, training := 0L]

        dataset_func[
            (azar <= PARAM$train_final$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
            training := 1L
        ]

        dataset_func[, azar:= NULL] # elimino la columna azar

        # target
        dataset_func[,clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)]

        # utilizo  zLightGBM  la nueva libreria
        if( !require("zlightgbm") ) install.packages("https://storage.googleapis.com/open-courses/dmeyf2025-e4a2/zlightgbm_4.6.0.99.tar.gz", repos= NULL, type= "source")
        require("zlightgbm")
        Sys.time()

        # canaritos
        

        cols0 <- copy(colnames(dataset_func))
        filas <- nrow(dataset_func)

        for( i in seq(PARAM$qcanaritos) ){
        dataset_func[, paste0("canarito_",i) := runif( filas) ]
        }

        # las columnas canaritos mandatoriamente van al comienzo del dataset
        cols_canaritos <- copy( setdiff( colnames(dataset_func), cols0 ) )
        setcolorder( dataset_func, c( cols_canaritos, cols0 ) )

        Sys.time()


        # los campos que se van a utilizar

        campos_buenos <- setdiff(
        colnames(dataset_func),
        c("clase_ternaria", "clase01", "training"))

        # dejo los datos en el formato que necesita LightGBM

        dtrain_final <- lgb.Dataset(
        data= data.matrix(dataset_func[training == 1L, campos_buenos, with= FALSE]),
        label= dataset_func[training == 1L, clase01],
        free_raw_data= FALSE
        )

        cat("filas", nrow(dtrain_final), "columnas", ncol(dtrain_final), "\n")
        Sys.time()
        
        # entreno el modelo

        modelo_final <- lgb.train(
        data= dtrain_final,
        param= PARAM$lgbm
        )

        Sys.time()

        # grabo el modelo generado, esto pude ser levantado por LighGBM en cualquier maquina
        lgb.save(modelo_final, file = paste0(nombre, "_zmodelo.txt"))

        # grabo un dataset que tiene el detalle de los arboles de LightGBM
        tb_arboles <- lgb.model.dt.tree(modelo_final)
        fwrite(tb_arboles, file = paste0(nombre, "_tb_arboles.txt"), sep="\t")

        cat("cantidad arbolitos=", tb_arboles[, max(tree_index)+1],"\n" )
        cat("summary de las hojas de los arboles")
        summary( tb_arboles[, list(hojas=max(leaf_index, na.rm=TRUE)+1), tree_index][,hojas])

        Sys.time()

        # aplico el modelo a los datos sin clase
        dfuture <- dataset[foto_mes %in% PARAM$future]

        # penosamente, en la versión actual de zLightGBM  los campos canaritos
        #  aunque no se utilizan para nada, también deben estar en el dataset donde se hace el predict()
        filas <- nrow(dfuture)

        for( i in seq(PARAM$qcanaritos) ){
        dfuture[, paste0("canarito_",i) := runif( filas) ]
        }

        prediccion <- predict(
        modelo_final,
        data.matrix(dfuture[, campos_buenos, with= FALSE]),
        )

        # tabla de prediccion, puede ser util para futuros ensembles
        #  ya que le modelo ganador va a ser un ensemble de LightGBMs

        tb_prediccion <- dfuture[, list(numero_de_cliente, foto_mes)]
        tb_prediccion[, prob := prediccion ]

        # grabo las probabilidad del modelo
        fwrite(tb_prediccion,
        file= paste0(nombre, "_prediccion.txt"),
        sep= "\t"
        )
    }        

In [ ]:
evaluar_desde_pred <- function(
  nombre,
  future_mes,
  dataset,                      # dataset con clase_ternaria ya construida
  premio_baja2 = 780000,
  costo_no_baja2 = -20000,
  usar_meseta = TRUE,
  ventana = 2001
){
  stopifnot(is.data.table(dataset))
  arch_pred <- paste0(nombre, "_prediccion.txt")
  if (!file.exists(arch_pred)) stop("No existe: ", arch_pred)

  # 1) leo predicciones (solo numero_de_cliente y prob)
  pred <- fread(arch_pred)
  req_cols <- c("numero_de_cliente","prob")
  if (!all(req_cols %in% names(pred))) {
    stop("El archivo debe tener columnas: ", paste(req_cols, collapse=", "))
  }

  # 2) verdad-terreno del mes futuro
  verdad <- dataset[foto_mes == future_mes, .(numero_de_cliente, clase_ternaria)]
  if (nrow(verdad) == 0) stop("No hay registros en dataset para foto_mes=", future_mes)

  # 3) merge por cliente (mucho ojo: si hay duplicados de cliente en pred, deduplicá antes)
  pred <- unique(pred, by="numero_de_cliente")
  tb <- merge(pred, verdad, by = "numero_de_cliente", all.x = TRUE)

  # 4) ganancia por fila
  tb[, gan := costo_no_baja2]
  tb[clase_ternaria == "BAJA+2", gan := premio_baja2]

  # 5) ranking por prob y cumsums
  setorder(tb, -prob)
  tb[, gan_acum := cumsum(gan)]

  # 6) mejor K crudo
  best_k_crudo    <- which.max(tb$gan_acum)
  best_gain_crudo <- tb$gan_acum[best_k_crudo]

  # 7) meseta opcional
  tb[, gan_meseta := NA_real_]
  best_k_meseta <- NA_integer_
  best_gain_meseta <- NA_real_
  if (usar_meseta) {
    if (!requireNamespace("zoo", quietly=TRUE)) install.packages("zoo")
    library(zoo)
    tb[, gan_meseta := zoo::rollapply(
      data = gan_acum, width = ventana, FUN = mean,
      align = "center", fill = NA, na.rm = TRUE
    )]
    best_k_meseta    <- which.max(tb$gan_meseta)
    best_gain_meseta <- tb$gan_meseta[best_k_meseta]
  }

  # 8) guardo curva y resumen (sin tocar *_prediccion.txt)
  fwrite(tb[, .(rank = .I, prob, gan, gan_acum, gan_meseta)],
         file = paste0(nombre, "_valid_curve.txt"), sep = "\t")

  resumen <- data.table(
    nombre = nombre,
    future_mes = future_mes,
    best_k_crudo = best_k_crudo,
    best_gain_crudo = best_gain_crudo,
    best_k_meseta = best_k_meseta,
    best_gain_meseta = best_gain_meseta
  )
  fwrite(resumen, file = paste0(nombre, "_valid_resumen.txt"), sep = "\t")

  # 9) importancia de features, si existe el modelo
  impo_path <- paste0(nombre, "_importancia.txt")
  modelo_path <- paste0(nombre, "_zmodelo.txt")
  importancia <- NULL
  if (file.exists(modelo_path)) {
    if (!requireNamespace("lightgbm", quietly=TRUE)) install.packages("lightgbm")
    library(lightgbm)
    modelo <- lgb.load(modelo_path)
    importancia <- lgb.importance(modelo)
    if (!is.null(importancia) && nrow(importancia)) {
      setDT(importancia)
      setorder(importancia, -Gain)
      fwrite(importancia, file = impo_path, sep = "\t")
    }
  }

  invisible(list(
    tabla = tb,
    resumen = resumen,
    importancia = importancia
  ))
}

In [ ]:
# training y future
Sys.time()

PARAM$train_final$meses <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104)
PARAM$train_final$undersampling <- 0.50

PARAM$future <- c(202106)

In [ ]:
# se filtran los meses donde se entrena el modelo final
dataset_valid06 <- dataset[foto_mes %in% PARAM$train_final$meses]

In [ ]:
correr_zlgbm(
  dataset_func = dataset_valid06,
  params = PARAM,
  nombre = "valid202106"
)


In [ ]:
res_eval <- evaluar_desde_pred(
  nombre = "valid202106",
  future_mes = 202106,
  dataset = dataset,     # el mismo donde generaste clase_ternaria
  usar_meseta = TRUE,
  ventana = 2001
)

K_opt <- if (!is.na(res_eval$resumen$best_k_meseta)) res_eval$resumen$best_k_meseta else res_eval$resumen$best_k_crudo
cat("K óptimo =", K_opt,
    "\nGanancia (crudo) =", res_eval$resumen$best_gain_crudo,
    "\nGanancia (meseta) =", res_eval$resumen$best_gain_meseta, "\n")

## Produccion

Las decisiones que se toman para la construccion del modelo final son:
* Los positvos son  POS={"BAJA+1", "BAJA+2"}, esta es una meticulosa decisión.
* Se entrena en los cuatro meses  {202101, 202102, 202103, 202104}, esta es una meticulosa decisión.
* Se hace un muy consevador undersampling del **0.50** = 50% de la clase mayoritaria (los "CONTINUA" )
* Obviamente los datos donde se aplica el modelo es el mes  {202106}
* Por experimentos en meses anteriores, se decide cortar en los 11000 registros con mayor probabildiad de POS={"BAJA+1", "BAJA+2"}, , esta es una *enorme* decisión.
* No se optmizan los hiperparámetros de LightGBM, sino que se llama a la libreria *zLightGBM*
* Para *zLightGBM*  se crean **100** canaritos, esta es una decisión enorme !

In [ ]:
# training y future
Sys.time()

PARAM$train_final$meses <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106)
PARAM$train_final$undersampling <- 0.50

PARAM$future <- c(202108)

### Final Training Strategy

Se entrena en los cuatro meses {202101, 202102, 202103, 202104}
<br>Se hace un conservador undersampling del 50% de la clase mayoritaria (los "CONTINUA" )

In [ ]:
# se filtran los meses donde se entrena el modelo final
dataset_train_final <- dataset[foto_mes %in% PARAM$train_final$meses]

In [ ]:
correr_zlgbm(
  dataset_func = dataset_train_final,
  params = PARAM,
  nombre = "final"
)


### Clasificacion

Se tomó la decisión de enviar a los 11000 registros con mayor probabilidad de POS={"BAJA+1","BAJA+"}
<br> esto se determinó en forma artesanal analizando meses anterior
<br> esta es una muy importante decisión 

In [ ]:
# genero archivos con los  "envios" mejores
dir.create("kaggle", showWarnings=FALSE)

# ordeno por probabilidad descendente

tb_pred_final <- fread("final_prediccion.txt")
setorder(tb_pred_final, -prob)

envios <- 11000
tb_pred_final[, Predicted := 0L] # seteo inicial a 0
tb_pred_final[1:envios, Predicted := 1L] # marco los primeros

archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

# grabo el archivo
fwrite(tb_pred_final[, list(numero_de_cliente, Predicted)],
  file= archivo_kaggle,
  sep= ","
)

### Subida a Kaggle

In [ ]:
# # subida automática a Kaggle, solo tiene sentido para la Primera Competencia

# comando <- "kaggle competitions submit"
# UBA_comp <- "-c dm-ey-f-2025-primera"
# arch <- paste("-f", archivo_kaggle)

# mensaje <- paste0("-m 'under=", PARAM$train_final$undersampling,
#   "  cana=", PARAM$qcanaritos,
#   "  gb=", PARAM$lgbm$gradient_bound,
#   "  ff=", PARAM$lgbm$feature_fraction,
#   "  mdil=", PARAM$lgbm$min_data_in_leaf,
#   "  lr=", PARAM$lgbm$learning_rate, "'"
# )s

# linea <- paste(comando, UBA_comp, arch, mensaje)
# salida <- system(linea, intern=TRUE)
# cat(salida)

In [ ]:
Sys.time()